In [1]:
import uuid, time
from src.spark.spark import get_spark_session
from src.utils.logging_config import setup_logging
from src.io.writer import DataFrameWriter

In [ ]:
layer = "gold"
dataset = "fact_payment_summary"

# Load Spark Session
spark = get_spark_session()

# Load logging
log = setup_logging(f"{dataset}_{layer}")

# Create logical variables
run_id = str(uuid.uuid4())

In [ ]:
log.info(f"[PARAMS] dataset={dataset} layer={layer}")

start = time.time()
log.info(f"[GOLD][START] run_id={run_id}")

try:

    writer = DataFrameWriter(spark)
    
    fact_sales = spark.sql("""
        WITH payments AS(
            SELECT
                op.order_id,
                sum(op.payment_value) AS total_payment_value,
                COUNT(DISTINCT op.payment_type) AS total_methods_distincts_payments,
                MAX(op.payment_installments) AS max_installments
            FROM olist.silver.order_payments op
            GROUP BY op.order_id
        ), multiple_methods_payments AS(
            SELECT mmp.order_id, mmp.payment_type FROM 
                (SELECT
                    op.order_id,
                    op.payment_type,
                    ROW_NUMBER() OVER (PARTITION BY op.order_id ORDER BY op.payment_value DESC) AS partiion_payment
                FROM olist.silver.order_payments op) mmp
                WHERE mmp.partiion_payment = 1
        )
        SELECT 
            op.order_id,
            op.total_payment_value,
            op.total_methods_distincts_payments,
            op.max_installments,
            mmp.payment_type AS primary_payment_type,
            c.customer_state
        FROM payments op
        LEFT JOIN multiple_methods_payments mmp ON mmp.order_id = op.order_id
        LEFT JOIN olist.silver.orders o ON o.order_id = op.order_id
        LEFT JOIN olist.silver.customers c ON c.customer_id = o.customer_id
        ORDER BY op.total_payment_value DESC, op.total_methods_distincts_payments DESC, c.customer_state
    """)
    
    writer.write_delta_batch(fact_sales, "olist", "gold", "fact_payment_summary", mode="overwrite")
    duration_s = round(time.time() - start, 3)
    log.info(f"[GOLD][SUCCESS] run_id={run_id} dataset={dataset} duration_s={duration_s}s")

except Exception as e:
    duration_s = round(time.time() - start, 3)
    log.error(f"[GOLD][FAILED] run_id={run_id} duration_s={duration_s} error={repr(e)}")
    raise